In [1]:
!pip install roboflow torch torchvision opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 141.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="KeqFZZO46zSUJr7Nvr51")
project = rf.workspace("jodaryle-factor-gjkmr").project("license-plate-object-detection")
version = project.version(6)
dataset = version.download("voc")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Object-Detection-6 in voc:: 100%|██████████| 891/891 [00:00<00:00, 8783.19it/s]


In [4]:
import torch
import cv2
import xml.etree.ElementTree as ET
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [5]:
class PlateDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        # Load all image files, sorting them to ensure they match annotations
        self.imgs = list(sorted([f for f in os.listdir(root) if f.endswith(".jpg")]))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, self.imgs[idx])
        xml_path = os.path.join(self.root, self.imgs[idx].replace(".jpg", ".xml"))

        img = Image.open(img_path).convert("RGB")

        # Parse Pascal VOC XML
        tree = ET.parse(xml_path)
        root = tree.getroot()
        boxes = []
        labels = []

        for obj in root.findall("object"):
            label = obj.find("name").text
            # Map your class names to integers (1 is license-plate, 0 is background)
            labels.append(1)

            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            boxes.append([xmin, ymin, xmax, ymax])

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [6]:
def get_model(num_classes):
    # Load a model pre-trained on COCO
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    # Get the number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Replace the pre-trained head with a new one
    # num_classes includes background (e.g., Background + Plate = 2)
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

In [9]:
from torchvision import transforms as T

def get_transform():
    return T.Compose([T.ToTensor()])

# Setup
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
dataset = PlateDataset('/content/License-Plate-Object-Detection-6/train', transforms=get_transform())
data_loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))

model = get_model(num_classes=2) # 1 class (plate) + background
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

# Training for 10 epochs
model.train()
for epoch in range(10):
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {losses.item()}")

torch.save(model.state_dict(), "faster_rcnn_plate.pth")

Epoch 0 Loss: 0.19003936648368835
Epoch 1 Loss: 0.12273349612951279
Epoch 2 Loss: 0.10898571461439133
Epoch 3 Loss: 0.061935584992170334
Epoch 4 Loss: 0.071773961186409
Epoch 5 Loss: 0.04949343577027321
Epoch 6 Loss: 0.05531490594148636
Epoch 7 Loss: 0.04088374972343445
Epoch 8 Loss: 0.11139433830976486
Epoch 9 Loss: 0.05460864678025246


In [10]:
import torch
import cv2
import numpy as np
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision import transforms as T
from PIL import Image

# --- STEP 1: Re-initialize the Model Architecture ---
def get_model(num_classes):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model(num_classes=2) # 1 class (plate) + background

# --- STEP 2: Load the Saved Weights ---
model.load_state_dict(torch.load("/content/faster_rcnn_plate.pth", map_location=device))
model.to(device)
model.eval() # Set to evaluation mode

# --- STEP 3: Run Detection on an Image ---
def detect_and_show(image_path, threshold=0.8):
    # Load image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess (Convert to Tensor)
    transform = T.Compose([T.ToTensor()])
    img_tensor = transform(img_rgb).to(device).unsqueeze(0) # Add batch dimension

    with torch.no_grad():
        prediction = model(img_tensor)

    # Extract results
    boxes = prediction[0]['boxes'].cpu().numpy()
    scores = prediction[0]['scores'].cpu().numpy()
    labels = prediction[0]['labels'].cpu().numpy()

    # Loop through detections
    for i in range(len(boxes)):
        if scores[i] > threshold:
            box = boxes[i].astype(int)
            # Draw Bounding Box
            cv2.rectangle(img, (box[0], box[1]), (box[2], box[3]), (0, 255, 0), 2)
            # Draw Label & Score
            text = f"Plate: {scores[i]:.2f}"
            cv2.putText(img, text, (box[0], box[1]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Save and display
    cv2.imwrite("result.jpg", img)
    print("Inference complete. Result saved as result.jpg")

# Test it on a new image
detect_and_show("/content/License-Plate-Object-Detection-6/test/1k9uinoun9ha1_jpg.rf.1d2c4d7443fc7b968eb78c92679aaa68.jpg")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 200MB/s]


Inference complete. Result saved as result.jpg
